# Activity: Fed-Batch mAb Production as a Markov Decision Process
In this graded activity, you will frame fed-batch production of a monoclonal antibody (mAb) in CHO cells as a sequential decision problem and solve it with reinforcement learning. Here is the scenario:

__Scenario__: A bioreactor is run as a _fed-batch_: starting from a partially filled vessel, each cycle you choose a glucose feed rate. Feeding adds volume and substrate, growing the culture and (in proportion to the cells) producing antibody — but it also dilutes the culture and triggers overflow metabolism that accumulates lactate. Too much lactate crashes the culture; meanwhile the reactor fills, and once it is full the batch is harvested. The challenge is to learn a feed policy that fills the reactor with a dense, productive, un-crashed culture, maximizing the antibody harvested.

This is the fed-batch counterpart of the [adaptive-dosing example](CHEME-153-M4-Example-AdaptiveDosing-Q-Learning-Ungraded-Codio-Activity.ipynb): the state is the culture condition, the actions are feed rates, a culture crash plays the role of toxic death, and the reward is the product harvested when the reactor fills.

> __Learning Objectives:__
>
> After completing this activity, students will be able to:
> * __Formulate a bioprocess as a Markov decision process:__ Encode biomass, lactate, and reactor volume as a state, feed rate as an action, and Monod growth with dilution and by-product inhibition as transitions.
> * __Solve a known MDP exactly and benchmark a learned policy:__ Compute the optimal feed policy by value iteration, then train a model-free Q-learning agent and verify it against the optimum.
> * __Verify a reinforcement-learning solution with automatic checks:__ Use the value-iteration optimum as a ground truth to confirm the learned policy harvests a productive batch where constant feeding cannot.

Each task ends with a cell of `@assert` checks; when a task is correct, that cell prints `Task N checks passed.`

Let's get started!
___

## The Markov Decision Process
We describe the culture with three state variables on a grid: the viable biomass $X\in[0,1]$, the accumulated lactate $L\in[0,1]$, and the reactor volume $V\in[0,1]$ (all normalized). Each cycle the agent applies a feed rate $f\in\{0, \tfrac{1}{3}, \tfrac{2}{3}, 1\}$. The feed adds volume $\Delta V = v_{f}\,f$ and sets the dilution rate $D = \Delta V / V$. The state then evolves by Monod growth with lactate inhibition and dilution, overflow lactate production with consumption and dilution, and rising volume:
$$
\begin{align*}
X_{t+1} &= X_{t} + \mu\,X_{t}(1 - X_{t}) - k_{d}\,X_{t} - D\,X_{t}, \qquad \mu = \mu_{\max}\,\frac{f}{K_{f}+f}\,\frac{K_{I}}{K_{I}+L_{t}}\\
L_{t+1} &= L_{t} + y_{L}\,f\,X_{t} - k_{L}\,L_{t} - D\,L_{t}\\
V_{t+1} &= V_{t} + \Delta V
\end{align*}
$$
The continuous next state is snapped to the nearest grid cell, making this a finite Markov decision process. There are two absorbing states:

* __Harvest__ when $V \geq V_{\max}$: the reactor is full and the batch is collected; the reward is the product harvested, $q_{P}\,X\,V$.
* __Culture crash__ (a negative terminal reward) when $L \geq L_{\text{crash}}$: the batch is lost.

The reward is _zero every cycle until harvest_ — you are paid only for what you collect when the reactor fills. Because feeding raises lactate, dilutes the culture, and fills the reactor toward harvest, the agent must pace the feed to reach harvest with a dense, un-crashed culture. This is a genuinely sequential decision problem.
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

  Activating 

This activity uses local source files: the environment and agent types are in [`src/Types.jl`](src/Types.jl), their constructors in [`src/Factory.jl`](src/Factory.jl), and the dynamics `world(...)`, the `solve(...)` routines (value iteration and Q-learning), `policy(...)`, and `simulate(...)` in [`src/Compute.jl`](src/Compute.jl). These are loaded for you by `Include.jl`.

## Task 1: Build the bioreactor environment
Build the fed-batch environment with the [`MyFedBatchBioreactorModel` type](src/Types.jl) using the parameters below, and save it in the `env::MyFedBatchBioreactorModel` variable. We use a $21\times21\times21$ state grid (biomass × lactate × volume) and four feed levels.

> __Parameters:__ growth `mumax`, feed half-saturation `Kf`, lactate inhibition `KIL`, death `kd`, lactate yield `yL`, lactate consumption `kL`, productivity `qP`; volume added per cycle at full feed `vfeed`, initial volume `V0`, harvest volume `Vmax`; the culture crashes at `Lcrash` with penalty `Rcrash`.

In [ ]:
env = build(MyFedBatchBioreactorModel, (
    nlevels = 21,                   # 21^3 state grid (X, L, V in 0, 0.05, ..., 1)
    feeds   = [0.0, 1/3, 2/3, 1.0], # none, low, medium, high

    mumax = 1.00, # maximum specific growth rate
    Kf    = 0.30, # feed half-saturation (Monod in feed)
    KIL   = 0.35, # lactate inhibition constant
    kd    = 0.03, # cell death rate
    yL    = 1.00, # lactate yield per unit feed * biomass (overflow)
    kL    = 0.30, # lactate consumption per cycle
    qP    = 1.0,  # specific productivity (harvest reward = qP * X * V)

    vfeed = 0.05, # reactor volume added per cycle at full feed
    V0    = 0.50, # initial (inoculation) volume
    Vmax  = 1.00, # harvest when the reactor is full

    Lcrash = 0.90, Rcrash = -25.0   # culture-crash threshold and penalty
));

nstates = env.nlevels^3;     # number of grid states
nfeeds  = length(env.feeds); # number of feed actions

In [ ]:
let
    @assert env.nlevels == 21
    @assert length(env.feeds) == 4
    @assert nstates == 9261
    @assert isterminal(env, stateindex(env, 0.5, 0.95, 0.7)) == true   # high lactate -> crashed
    @assert isterminal(env, stateindex(env, 0.5, 0.2,  1.0)) == true   # full reactor -> harvested
    @assert isterminal(env, stateindex(env, 0.5, 0.2,  0.7)) == false  # mid-batch
    println("Task 1 checks passed.")
end

## Why a constant feed fails
Before solving, let's see what happens under the four _constant_ feed schedules, starting from a freshly inoculated reactor $(X_0, L_0, V_0) = (0.10, 0.0, 0.50)$.

> A constant policy returns the same feed in every state. `simulate(...)` rolls it out and reports the harvested `product`, the `outcome` (`:harvest`, `:crash`, or `:timeout`), and the number of cycles.

Run the baselines:

In [ ]:
baseline_best = let
    s0 = stateindex(env, 0.10, 0.0, env.V0)
    df = DataFrame(); best = 0.0
    for (name, a) in [("none",1), ("low",2), ("medium",3), ("high",4)]
        r = simulate(env, fill(a, nstates), s0)
        push!(df, (schedule=name, feed=env.feeds[a], outcome=String(r.outcome),
                   cycles=r.steps, product=round(r.product, digits=3)))
        best = max(best, r.product)
    end
    pretty_table(df; backend=:text, table_format=TextTableFormat(borders=text_table_borders__compact))
    best
end;

No constant feed harvests any product: feeding too little never fills the reactor (the batch times out), while medium or high feed crashes the culture on lactate before it can fill. The agent needs a feed _policy_ that adapts to the culture state.
___

## Task 2: The optimal feed policy by value iteration
Because the dynamics in `world(...)` are known, compute the optimal policy exactly with __value iteration__. Call `solve(...)` on the environment to get the optimal action-value table `Qstar`, extract the optimal policy with `policy(...)`, and roll it out from the inoculum.

In [ ]:
Qstar  = solve(env; γ = 0.95);  # value iteration (model-based optimum)
π_star = policy(Qstar);         # optimal feed policy
vi = simulate(env, π_star, stateindex(env, 0.10, 0.0, env.V0));
println("Optimal feed policy: ", vi.outcome, " in ", vi.steps, " cycles, product = ", round(vi.product, digits=3))

In [ ]:
let
    @assert size(Qstar) == (nstates, nfeeds)
    @assert vi.outcome == :harvest   # the optimal policy fills the reactor and harvests
    @assert vi.product > 0.3         # with a productive batch (constant feeds harvest nothing)
    println("Task 2 checks passed.")
end

## Task 3: Learn the feed policy from experience (Q-learning)
Now solve the same problem _model-free_: the agent only sees sampled transitions $(s, a, r, s')$ and must learn a policy. Build a [`MyQLearningAgentModel`](src/Types.jl), train it with `solve(...)`, extract the learned policy, and roll it out.

> Training runs many episodes from random culture states; the agent updates its $Q$ table with a $1/N(s,a)$ learning-rate schedule, which satisfies the convergence conditions from the lecture. This cell runs two million episodes over the $21^3$ states and may take several seconds.

In [ ]:
agent = build(MyQLearningAgentModel, (
    states  = collect(1:nstates),
    actions = collect(1:nfeeds),
    α = 0.1,                     # nominal rate (the solver anneals as 1/N(s,a))
    γ = 0.95,                    # discount factor (matches value iteration)
    Q = zeros(nstates, nfeeds)   # initialize the Q-table to zeros
));

agent = solve(agent, env; episodes = 2_000_000, maxsteps = 80); # model-free training
π_q = policy(agent.Q);
ql = simulate(env, π_q, stateindex(env, 0.10, 0.0, env.V0));
println("Learned feed policy: ", ql.outcome, " in ", ql.steps, " cycles, product = ", round(ql.product, digits=3))

In [ ]:
let
    @assert ql.outcome == :harvest   # the learned policy fills the reactor and harvests
    @assert ql.product > 0.3         # with a productive batch (constant feeds harvest nothing)
    println("Task 3 checks passed.")
end

## Task 4: Benchmark the learned policy against the optimum
Finally, compare the model-free policy to the exact value-iteration optimum. Compute the product ratio (how much of the optimal harvest the learned policy captures) and confirm it lands within tolerance.

In [ ]:
ratio = ql.product / vi.product;
println("Q-learning captured ", round(100*ratio, digits=1), "% of the value-iteration optimum")
println("(harvest ", round(ql.product, digits=3), " vs ", round(vi.product, digits=3),
        "; best constant feed = ", round(baseline_best, digits=3), ")")

In [ ]:
let
    @assert ratio ≥ 0.6                # learned policy is within 40% of the exact optimum
    @assert ql.product > baseline_best # and strictly better than any constant feed
    println("Task 4 checks passed.")
end

## Visualizing the optimal batch
The plot below traces the optimal batch from inoculation to harvest: biomass and lactate rise as the agent feeds, the reactor volume climbs toward the harvest line, and the feed is eased whenever lactate approaches the crash line — a pulse-and-fill strategy.

`Unhide` the code block below to see how we build the plot:

In [ ]:
let
    cycles = 0:(length(vi.X)-1)
    p = plot(cycles, vi.X; lw=2, marker=:circle, ms=3, label="biomass X",
        xlabel="feed cycle", ylabel="normalized", title="Optimal fed-batch (value iteration)")
    plot!(p, cycles, vi.L; lw=2, marker=:square,  ms=3, label="lactate L")
    plot!(p, cycles, vi.V; lw=2, marker=:diamond, ms=3, label="volume V")
    plot!(p, cycles, vi.feed; seriestype=:steppost, lw=2, ls=:dash, label="feed f")
    hline!(p, [env.Lcrash]; color=:red,   ls=:dot, label="crash")
    hline!(p, [env.Vmax];   color=:green, ls=:dot, label="harvest")
    p
end

The state is 3-D, so we can't show the whole policy at once. The heatmap below is a slice at a nearly-full reactor ($V \approx 0.85$): the optimal feed in each $(X, L)$ state. The agent feeds when lactate is low and backs off as lactate approaches the crash line.

In [ ]:
let
    Vslice = 0.85
    kV = round(Int, Vslice/env.step) + 1
    Xgrid = [(i-1)*env.step for i in 1:env.nlevels]
    Lgrid = [(j-1)*env.step for j in 1:env.nlevels]

    M = fill(NaN, env.nlevels, env.nlevels)
    for iX in 1:env.nlevels, iL in 1:env.nlevels
        s = env.states[(iX, iL, kV)]
        isterminal(env, s) == false && (M[iL, iX] = env.feeds[π_star[s]])
    end

    p = heatmap(Xgrid, Lgrid, M; c=:viridis, colorbar_title="optimal feed",
        xlabel="biomass X", ylabel="lactate L", title="Optimal feed at V = $(Vslice)")
    hline!(p, [env.Lcrash]; color=:red, ls=:dash, label="culture crash")
    p
end

## Summary
In this activity, we framed fed-batch mAb production — with reactor volume, dilution, and a harvest at the end of the batch — as a Markov decision process, and solved it two ways: exactly with value iteration and model-free with Q-learning, then benchmarked the learned policy against the optimum.

> __Key Takeaways:__
>
> * __Sequential structure makes it a reinforcement learning problem:__ Feeding raises lactate, dilutes the culture, and fills the reactor toward harvest, so the action shapes the entire future of the batch. No constant feed harvests any product.
> * __Value iteration gives the exact optimum:__ With the dynamics known, value iteration computes the optimal feed schedule and the product it harvests, which serves as a ground-truth benchmark.
> * __Model-free learning matches the optimum:__ Q-learning, using only sampled experience, learns a feed policy that fills the reactor with a dense, un-crashed culture and harvests essentially the same product as the exact optimum, where every constant feed fails.

When the dynamics are known, value iteration gives the exact optimal policy; when they are not, model-free reinforcement learning can recover it from experience alone.
___